<a href="https://colab.research.google.com/github/maryamyazdi/Sentiment-Analysis/blob/main/Copy_of_new2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install hazm
import hazm as hz
import re
import pickle
import os
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

     |████████████████████████████████| 316 kB 5.4 MB/s 
     |████████████████████████████████| 233 kB 42.9 MB/s 
     |████████████████████████████████| 1.4 MB 43.0 MB/s 
  Created wheel for nltk: filename=nltk-3.3-py3-none-any.whl size=1394485 sha256=5b84d06c43955bcb43fb653df8c8e43e4b2401f96099ae0f195f4839ecaea254
  Stored in directory: /root/.cache/pip/wheels/9b/fd/0c/d92302c876e5de87ebd7fc0979d82edb93e2d8d768bf71fac4
  Created wheel for libwapiti: filename=libwapiti-0.2.1-cp37-cp37m-linux_x86_64.whl size=154624 sha256=e09495f8c70b837c469a79cb14b675349d7aea3ee017b5f86798a1c510f5d520
  Stored in directory: /root/.cache/pip/wheels/ab/b2/5b/0fe4b8f5c0e65341e8ea7bb3f4a6ebabfe8b1ac31322392dbf
Successfully built nltk libwapiti
  Attempting uninstall: nltk
    Found existing installation: nltk 3.2.5
    Uninstalling nltk-3.2.5:
      Successfully uninstalled nltk-3.2.5
Mounted at /content/drive


In [220]:
df = pd.read_csv('/content/drive/My Drive/train.csv', sep='\t', index_col=0)
train_data = (df.head(n=300)).drop(axis=1, columns='label')

def fixup(x):
    x = x.replace('\u200c', '').replace('\xa0','').replace('\r\n',' ').replace('|',' ')
    return x

normalizer = hz.Normalizer()
def my_tokenizer(text):
  text = re.sub(r"[\{\}\؛\*\=\-\+\/\n\(\)]"," ",str(text))
  text = re.sub("[ ]+"," ",text)
  text = re.sub("\!+","!",text)
  # is ? and ؟ same?
  text = re.sub("[؟]+","؟",text)
  text = re.sub("\?+","?",text)
  text = re.sub("[.]+",".",text)
  #text = re.sub("[ر]+","ر",text)
  text = fixup(normalizer.normalize(text))
  sentences = hz.sent_tokenize(text)
  words = []
  for every_sentence in sentences:
    words += hz.word_tokenize(every_sentence)
  return words

train_data['comment'] = train_data['comment'].apply(my_tokenizer)
# tokens = []
# num_words = []

# for comment in train_data['comment']:
#   tokens.append(my_tokenizer(comment))
#   num_words.append(len(tokens[-1]))

In [221]:
len(train_data['comment'][0])

9

In [209]:
  # print statistics
print('Min length =', min(num_words))
print('Max length =', max(num_words))
print('Mean = {:.2f}'.format(np.mean(num_words)))
print('Std  = {:.2f}'.format(np.std(num_words)))
print('mean + 2 * sigma = {:.2f}'.format(np.mean(num_words) + 2.0 * np.std(num_words)))

Min length = 4
Max length = 126
Mean = 17.60
Std  = 16.04
mean + 2 * sigma = 49.68


In [18]:
class Vocabulary(object):
    
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.word2index = {}
        self.word2count = {}
        self.index2word = {}
        self.count = 0
    
    def add_word(self, word):
        if not word in self.word2index:
            self.word2index[word] = self.count
            self.word2count[word] = 1
            self.index2word[self.count] = word
            self.count += 1
        else:
            self.word2count[word] += 1
    
    def add_sentence(self, sentence):
        for word in self.tokenizer(sentence):
            self.add_word(word)

In [19]:
PAD = '<pad>'
UNK = '<unk>'

class TextClassificationDataset(Dataset):

    def __init__(self, tokenizer, vocab_path, max_len , min_count):
  
        self.vocab_path = vocab_path
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.min_count = min_count
        
        self.cache = {}
        self.vocab = None
        
        self.classes = []
        self.class_to_index = {}

        self.cache = {}
        self.vocab = None
        
        self.classes = []
        self.class_to_index = {}
        self.text_files = []
        
        split_path = f'{path}/{split}'
        
        for cls_idx, label in enumerate(os.listdir(split_path)):
            text_files = [(fname, cls_idx) for fname in glob(f'{split_path}/{label}/*.txt')]
            self.text_files += text_files
            self.classes += [label]
            self.class_to_index[label] = cls_idx
        
        self.num_classes = len(self.classes)
            
        # build vocabulary from training and valida
            
        # build vocabulary from training and validation texts
        self.build_vocab()
        
    def __getitem__(self, index):
        # read the tokenized text file and its label (neg=0, pos=1)
        # fname, class_idx = self.text_files[index]
        # if fname in self.cache:
          #return self.cache[fname], class_idx
          # print("done")
          # return

        # read text file 
        # text = open(fname).read()
        text = train_data['comment'][0]

        # tokenize the text file
        tokens = self.tokenizer(text)
        
        # padding and trimming
        if len(tokens) < self.max_len:
            num_pads = self.max_len - len(tokens)
            tokens = [PAD] * num_pads + tokens
        elif len(tokens) > self.max_len:
            tokens = tokens[:self.max_len]
            
        # numericalizing
        ids = torch.LongTensor(self.max_len)
        for i, word in enumerate(tokens):
            if word not in self.vocab.word2index:
                ids[i] = self.vocab.word2index[UNK]  # unknown words
            elif word != PAD and self.vocab.word2count[word] < self.min_count:
                ids[i] = self.vocab.word2index[UNK]  # rare words
            else:
                ids[i] = self.vocab.word2index[word]
                
        # save in cache for future use
        # self.cache[fname] = ids
        
        return ids
    
    def build_vocab(self):
        if not os.path.exists(self.vocab_path):
            vocab = Vocabulary(self.tokenizer)
            with open('/content/drive/My Drive/train.csv', encoding='utf8') as f:
              for (i, line) in enumerate(f):
                vocab.add_sentence((line.split(sep='\t'))[1])
                if i > 2000:
                  break

            # sort words by their frequencies
            words = [(0, PAD), (0, UNK)]
            words += sorted([(c, w) for w, c in vocab.word2count.items()], reverse=True)

            self.vocab = Vocabulary(self.tokenizer)
            for i, (count, word) in enumerate(words):
                self.vocab.word2index[word] = i
                self.vocab.word2count[word] = count
                self.vocab.index2word[i] = word
                self.vocab.count += 1

            pickle.dump(self.vocab, open(self.vocab_path, 'wb'))
        else:
            self.vocab = pickle.load(open(self.vocab_path, 'rb'))

In [ ]:
min_count = 10
vocab_path = 'vocab.pkl'
max_len = np.mean(num_words) + 2.0 * np.std(num_words)
train_ds = TextClassificationDataset(my_tokenizer, vocab_path, max_len, min_count)

vocab = train_ds.vocab
freqs = [(count, word) for (word, count) in vocab.word2count.items() if count >= min_count]
vocab_size = len(freqs) + 2  # for PAD and UNK tokens
print(f'Vocab size = {vocab_size}')

print('\nMost common words:')
for c, w in sorted(freqs, reverse=True)[:10]:
    print(f'{w}: {c}')

In [41]:
train_ds

[]

In [ ]:
!wget 'http://vectors.nlpl.eu/repository/20/61.zip' -O '/content/drive/MyDrive/w2vec.zip'

In [ ]:
!unzip /content/drive/MyDrive/w2vec.zip -d /content/drive/MyDrive/w2vec

In [21]:
import gensim
w2v_model = gensim.models.KeyedVectors.load_word2vec_format('/content/drive/MyDrive/w2vec/model.txt', binary=False, unicode_errors='replace')
w2v_weights = torch.FloatTensor(w2v_model.vectors)

In [ ]:
w2v_model.get_vector("علی")

In [222]:
train_data

,comment,label_id
0,"[واقعا, حیف, وقت, که, بنویسم, سرویس, دهیتون, ش...",1
1,"[قرار, بود, ۱, ساعته, برسه, ولی, نیم, ساعت, زو...",0
2,"[قیمت, این, مدل, اصلا, با, کیفیتش, سازگاری, ند...",1
3,"[عالللی, بود, همه, چه, درست, و, به, اندازه, و,...",0
4,"[شیرینی, وانیلی, فقط, یک, مدل, بود, .]",0
...,...,...
295,"[فقط, ساندویچ, سرد, شده_بود, ،, با, اینکه, ده,...",0
296,"[برگش, خیلی, عالی, بود, ولی, کباب, لقمهاش, بد,...",1
297,"[قبلا, خیلی, خوب, بود, ولی, نمیدونم, چرا, اینق...",1
298,"[زرشک, پلو, با, مرغ, با, بوی, بسیار, بد]",1


In [223]:
len(train_data['comment'][0])

9

In [224]:
max_len = 50
PAD = '<pad>'

def padding_and_trimming(tokens):
  if len(tokens) < max_len:
      num_pads = max_len - len(tokens)
      tokens = [PAD] * num_pads + tokens
  elif len(tokens) > max_len:
      tokens = tokens[:max_len]
  return tokens

In [225]:
train_data['comment'] = train_data['comment'].apply(padding_and_trimming)

In [226]:
len(train_data['comment'][0])

50

In [ ]:
train_data['comment'][1]

In [227]:
def s2v(s):
  l = []
  for w in s:
    if w == PAD:
      l.append(np.zeros(100))
      continue
    try:
      l.append(w2v_model.get_vector(w))
    except KeyError:
      l.append(np.ones(100))
  return l

In [228]:
train_data['comment'] = train_data['comment'].apply(s2v)

In [229]:
len(train_data['comment'][0])

50

In [257]:
import torch.nn as nn


class LSTMClassifier(nn.Module):
  def __init__(self, hidden_size, batch_size, layers_count):
    super(LSTMClassifier, self).__init__()
    self.hidden_size = hidden_size
    self.batch_size = batch_size
    # self.vocab_size = vocab_size
    self.layers_count = layers_count

    # self.embedding = nn.Embedding.from_pretrained(w2v_weights)
    self.embedding = nn.Embedding(90000, 100)
    self.lstm = nn.LSTM(self.embedding.embedding_dim, hidden_size, layers_count, dropout=0.2, bidirectional=False)
    self.classifier_layer = nn.Sequential(
        nn.Linear(hidden_size, 100),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(100, 2)
    )
    self.hidden = self.init_hidden()

  def init_hidden(self):
    h = torch.autograd.Variable(torch.zeros(self.layers_count, self.batch_size, self.hidden_size))
    c = torch.autograd.Variable(torch.zeros(self.layers_count, self.batch_size, self.hidden_size))
    return h, c

  def forward(self, x):
    print(x.size())
    x = self.embedding(x)
    x, self.hidden = self.lstm(x, self.hidden)
    x = self.classifier_layer(x[-1])
    return x


In [258]:
lstm_model = LSTMClassifier(hidden_size=256, batch_size=1, layers_count=1)

# is_gpu_enabled = torch.cuda.is_available()
is_gpu_enabled = False
if is_gpu_enabled:
  lstm_model = lstm_model.cuda()

/usr/local/lib/python3.7/dist-packages/torch/nn/modules/rnn.py:65: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  "num_layers={}".format(dropout, num_layers))


In [121]:
criterion = nn.CrossEntropyLoss()
if is_gpu_enabled:
  criterion = criterion.cuda()

optimizer = torch.optim.SGD(lstm_model.parameters(), lr=0.001, momentum=0.9)
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)

In [240]:
PAD = '<pad>'
UNK = '<unk>'

class TDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        # self.X_train = torch.tensor(dataset['comment'].values,dtype=torch.float32)
        # self.y_train = torch.tensor(dataset['label_id'].values,dtype=torch.float32)
        self.X_train = dataset['comment']
        self.y_train = dataset['label_id']
        self.max_len = 50
        

    def __len__(self):
        return len(self.X_train)

    def __getitem__(self, index):
        tokens = self.X_train[index]
        # print(len(self.X_train[index]))
        print('Index:', index)
        return tokens, self.y_train[index]
dataset = TDataset(train_data)

In [ ]:
dataset[0]

In [262]:
losses = []
for epoch in range(5):
  for i, (inputs, targets) in enumerate(dataset):
    # print(inputs)
    # break
    # print('xx', len(inputs))
    inputs = torch.autograd.Variable(torch.tensor(inputs, dtype=torch.int64))
    # break
    targets = torch.autograd.Variable(torch.tensor(targets, dtype=torch.int64))
    # break
    # print('len(in):', len(inputs))
    # print('len(tar):', len(targets))
    outputs = lstm_model(inputs)
    loss = criterion(outputs, targets)
    losses += loss.data[0]

    loss.backward()

    optimizer.step()

    if (i + 1) % 50 == 0:
      print(f'Epoch {epoch + 1}/5 : step {i + 1}/{len(train_ds) // 100}, loss: {loss.data[0]}')


Index: 0
torch.Size([50, 100])


IndexError: ignored

In [261]:
!pip install transformers
from transformers import AutoTokenizer, AutoModelForMaskedLM

Traceback (most recent call last):
  File "/usr/local/lib/python3.7/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 3021, in _dep_map
    return self.__dep_map
  File "/usr/local/lib/python3.7/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 2815, in __getattr__
    raise AttributeError(attr)
AttributeError: _DistInfoDistribution__dep_map

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.7/dist-packages/pip/_internal/cli/base_command.py", line 180, in _main
    status = self.run(options, args)
  File "/usr/local/lib/python3.7/dist-packages/pip/_internal/cli/req_command.py", line 199, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.7/dist-packages/pip/_internal/commands/install.py", line 385, in run
    conflicts = self._determine_conflicts(to_install)
  File "/usr/local/lib/python3.7/dist-packages/pip/_internal/commands/install.py", line 515, in _det

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModelForMaskedLM.from_pretrained("xlm-roberta-base")


Downloading:   0%|          | 0.00/512 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/4.83M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.04G [00:00<?, ?B/s]